# Worked solutions · Section 11 of the sample questions

The sample questions are in `documentation/tex/questions`.
For each question the entry gives the answer,
a solution that cites the lecture notes for every formula it uses,
and the misreading that leads to each of the other choices.
The facts that several questions use are stated once, at the head of the section,
as F1–F4.

A citation names the number of a statement in the current build of the notes
and its label, which survives renumbering:
Proposition A.5.2 (`prop.lowestSetBit`), Section 1.2.4 (`sec.fasterBook`).
Prices are integer counts of ticks.
A binary numeral is written as Python prints it, `0b` and then the digits, position 0 on the right.

The code cells are a calculator: arithmetic on the numbers of the question,
as in the examination.
No snippet is run here;
the notebook `11-binary-tricks` runs them and verifies every key.

In [ ]:
from math import ceil

# Section 11 · Integers in binary

**F1. A side as an integer.**
A non-negative integer is a set of positions: bit $k$ is set exactly when $k$ belongs to the set,
Section A.1 (`sec.numberAsSet`).
`b | (1 << k)` adds $k$ and `b & ~(1 << k)` removes it, Table A.1 (`tab.bitsAsSets`).
`TickArrayBook` keeps a bid at $p$ as bit $p - p_0$, counted up from the floor $p_0$,
and an ask at $p$ as bit $(p_0 + W - 1) - p$, counted down from the ceiling,
so that on both sides the best price is the highest set bit, Section 1.2.4 (`sec.fasterBook`).
`BitmapBook` counts both sides up from its floor, `1 << (price - self.floor)`, and has no ceiling:
its best ask is the lowest set bit.

**F2. The highest and the lowest set bit.**
For $b \geq 0$, `b.bit_length()` is $\deg \mathrm{bin}(b) + 1$, Lemma A.5.1 (`lemma.bitLengthIsDegree`),
with the convention $\deg 0 = -1$ of Section A.2 (`sec.bitSequences`):
`b.bit_length() - 1` is the position of the highest set bit.
For $b > 0$, $b \;\&\; (-b) = 2^{\nu(b)}$, where $\nu(b)$ is the number of zeros below the lowest set bit,
so the lowest set bit is `(b & -b).bit_length() - 1`, Proposition A.5.2 (`prop.lowestSetBit`).
At $b = 0$, $0 \;\&\; (-0) = 0$ and `(0).bit_length()` is 0, so both formulae return $-1$:
in a book, a position below the edge of the band, returned without an error
(the paragraph after Listing A.4 (`listing.lowestSetBit`)).

**F3. Negative integers.**
By Definition A.4.1 (`def.twosComplementSequence`) a negative integer $x$ is the sequence of bits
$\neg\,\mathrm{bin}(-1 - x)$, eventually one.
Python's `~x` is $\neg x$, and $\neg x = -x - 1$ on the whole of $\mathbb Z$
(Section A.4 (`sec.integersAsSequences`), after Proposition A.4.2 (`prop.integersAsSequences`)).
Hence $-x = \neg(x - 1)$: for $x = 2^k$, $-2^k = \neg(2^k - 1)$ has every bit from $k$ upwards set
and the $k$ bits below it clear,
while $\neg 2^k = -2^k - 1$ has bit $k$ alone clear.
The `&` of a non-negative integer with a negative one is non-negative,
the leading ones of the second meeting the leading zeros of the first.

**F4. The matching.**
`unito26.lob.orderbook.AggregateBook.submit`, quoted in 11.1, takes the best price of the opposite side
while the order has size left and that price is within its limit,
and writes every change of a size through `set_size`.
For a limit order this is the decomposition of Proposition 1.1.11 (`prop.decompositionOfLimitOrder`):
a market part, then what is left resting at the limit price.
A market buy carries the price `sys.maxsize` and a market sell the price 0,
so the limit stops neither;
what is left of a market order that has traded rests at the price of its last fill,
and a withdrawal writes the size it leaves through `set_size` as well,
Section 1.2.3 (`sec.foldImplementation`).

## 11.1 · `question.tickArrayIntegersAfterACrossingBuy`

A `TickArrayBook` with floor 200 and width 10;
bids 50 at 203, 40 at 202 and 70 at 200; asks 60 at 205, 30 at 206 and 90 at 208;
a limit buy of 100 at 206.

**Answer: (C).** `77 2 206 208`.

**The integers before.**
The ceiling is $200 + 10 - 1 = 209$.
By F1 the bids are bits 3, 2 and 0, counted up from the floor,
and the asks bits $209 - 205 = 4$, 3 and 1, counted down from the ceiling.
The bid integer is $2^3 + 2^2 + 2^0 = 13$ and the ask integer $2^4 + 2^3 + 2^1 = 26$.

**The matching.**
By Proposition 1.1.11 (`prop.decompositionOfLimitOrder`) the buy is a market buy of
$q_M = \min(100,\ 60 + 30) = 90$, the size of the asks at or below 206,
followed by a limit buy of $100 - 90 = 10$ at 206 (F4).
- The first pass finds the highest set bit of 26, bit 4, the ask $209 - 4 = 205$.
  It takes the 60 there, and the write of size 0 clears bit 4: the ask integer is 10, `0b1010`.
- The second finds bit 3, the ask 206, takes the 30, and clears bit 3: the ask integer is 2.
- The third finds bit 1, the ask $209 - 1 = 208$, above the limit, and the loop stops with 10 left.

The 10 rest at 206, bit $206 - 200 = 6$ of the bids: the bid integer is $13 \,|\, 2^6 = 77$, `0b1001101`.

**The best prices.**
The highest set bit of 77 is bit 6, the bid $200 + 6 = 206$;
that of 2 is bit 1, the ask $209 - 1 = 208$ (F2).
The book is not crossed: $206 < 208$.

In [ ]:
ceiling = 200 + 10 - 1
bids = 2**(203 - 200) | 2**(202 - 200) | 2**(200 - 200)
asks = 2**(ceiling - 205) | 2**(ceiling - 206) | 2**(ceiling - 208)
asks_after = asks & ~(2**(ceiling - 205)) & ~(2**(ceiling - 206))
bids_after = bids | 2**(206 - 200)
ceiling, bids, asks, asks & ~(2**(ceiling - 205)), asks_after, bids_after, bin(bids_after), 200 + bids_after.bit_length() - 1, ceiling - (asks_after.bit_length() - 1)

**The other choices.**
- (A) `77 256 206 208`: the asks of `TickArrayBook` taken to be held as `BitmapBook` holds them,
  counted up from the floor with the touch at the lowest set bit, the matching unchanged:
  the ask left at 208 would be bit 8, the integer 256.
- (B) `77 4 206 208`: the ceiling taken as $p_0 + W = 210$, one tick too high:
  the ask at 208 would be bit 2, the integer 4.
- (D) `13 2 203 208`: the resting part of the decomposition dropped, the 10 left after the fills discarded:
  the bids stay at 13, and the best bid at 203.
- (E) `77 26 206 205`: the buy rested whole at its limit without trading:
  the asks untouched, and a book crossed at 206 against 205, which the matching never leaves.

In [ ]:
2**(208 - 200), 2**(200 + 10 - 208)

## 11.2 · `question.bitmapBestAskFromTheWrongEnd`

A `BitmapBook` with floor 95; bids 40 at 98 and 60 at 97; asks 30 at 101, 50 at 102 and 20 at 105;
then a market buy of 100, the whole ask side.
A reads every best price as the highest set bit, B reads the best ask as the lowest;
neither tests for an empty side.

**Answer: (B).** A prints `105 94 -4`; B prints `101 94 -4`.

**The integers.**
`BitmapBook` counts both sides up from the floor (F1):
the bids are bits 3 and 2, the integer 12,
and the asks bits 6, 7 and 10, the integer $2^6 + 2^7 + 2^{10} = 1216$.

**Before.**
The best ask is the lowest ask, 101, which is bit 6, the lowest set bit of 1216.
- A returns $95 + \deg 1216 = 95 + 10 = 105$, the highest ask (F2).
- B returns the lowest set bit: $1216 \;\&\; (-1216) = 2^6$, whose `bit_length` is 7,
  and $95 + 7 - 1 = 101$, Proposition A.5.2 (`prop.lowestSetBit`).

**The market buy.**
Its price, `sys.maxsize`, stops no pass of the loop (F4).
A takes 20 at 105, 50 at 102 and 30 at 101, filling from the top of the side;
B takes the same sizes from the bottom.
In both, the third fill brings the total to 100 and empties the side:
the loop stops with nothing left, and nothing rests.

**After.**
The ask integer is 0.
In A, `(0).bit_length()` is 0; in B, $0 \;\&\; (-0) = 0$, whose `bit_length` is 0.
Both return $95 + 0 - 1 = 94$ (F2): a price below the bid of 98, returned without an error.
The best bid is the highest set bit of 12 in both, bit 3, the price 98,
and the spread is $94 - 98 = -4$.

In [ ]:
bids = 2**(98 - 95) | 2**(97 - 95)
asks = 2**(101 - 95) | 2**(102 - 95) | 2**(105 - 95)
bids, asks, 95 + asks.bit_length() - 1, asks & -asks, 95 + (asks & -asks).bit_length() - 1, 95 + bids.bit_length() - 1, 95 + (0 & -0).bit_length() - 1, 94 - 98

**The other choices.**
- (A) Both `101 None None`: both variants read as the package's `best_price`,
  which tests `if not bits` and finds the lowest ask.
  The verification notebook prints this for `BitmapBook` itself.
- (C) `105 None None` and `101 None None`: A's wrong end seen,
  and an empty integer taken to give `None`; neither variant tests for it.
- (D) Both `101 94 -4`: A's highest set bit taken for the best ask, as in `TickArrayBook`;
  counted up from the floor it is the highest ask.
- (E) `105 95 -3` and `101 95 -3`: the empty side read as bit 0, the floor, as though $\deg 0$ were 0;
  by the convention $\deg 0 = -1$ both formulae return $-1$.

## 11.3 · `question.occupancyWrittenByAddition`

A `TickArrayBook` with floor 95 and width 12; bids 100 at 99 and 60 at 98; a market sell of 40.
A writes the bit with `+=` and `-=`;
B flips it with `^=` when the price passes between empty and occupied.

**Answer: (D).** A prints `100 0`; B prints `99 60`.

**The writes before the sell.**
The bids are bits $99 - 95 = 4$ and 3 (F1).
Each is written to a clear bit, where addition and `|` agree, and where B flips the bit from 0 to 1:
in both books the integer is $16 + 8 = 24$, `0b11000`.

**The sell.**
The best bid is 99, bit 4.
A market sell carries the price 0, which stops no pass of the loop (F4):
it takes 40 of the 100 at 99, writes `set_size(BUY, 99, 60)`, and stops with nothing left.
- In A the size 60 is positive, so `+= 16` is applied to a set bit:
  $24 + 16 = 40$, `0b101000`.
  The carry clears bit 4 and sets bit 5.
  The highest set bit is 5, the price $95 + 5 = 100$, where nothing rests:
  the list holds 0 at 100 and 60 at 99,
  and `best_bid_size` reads it at the best bid: `100 0`.
- In B, `(100 > 0) != (60 > 0)` is false: a partial fill leaves the price occupied,
  the bit is not flipped, and the integer stays 24: `99 60`.

Addition agrees with `|` only on a clear bit, and subtraction with `& ~` only on a set one:
$1 + 1 = 2$ carries a digit (the paragraph after Proposition A.2.1 (`prop.binaryDigits`)).

In [ ]:
bits = 2**(99 - 95) + 2**(98 - 95)
bits, bits + 2**(99 - 95), bits | 2**(99 - 95), 95 + (bits + 2**(99 - 95)).bit_length() - 1

**The other choices.**
- (A) Both `99 60`: `+=` read as `|=`, the carry on a set bit missed.
- (B) `99 60` and `98 60`: `+=` read as `|=`, and B taken to flip the bit at every write:
  the partial fill would clear bit 4 and leave bit 3, the price 98, which holds 60.
- (C) `100 60` and `99 60`: the carry seen, and `best_bid_size` taken for the size just written, 60 at 99;
  it reads the list at the best bid, 100, which holds 0.
- (E) `100 0` and `98 60`: A right, and B flipped at every write, as in (B).

## 11.4 · `question.occupancyClearedWithMinusForTilde`

A `TickArrayBook` with floor 95 and width 12; bids 100 at 99, 60 at 98 and 40 at 96;
a withdrawal of the 60 at 98.
The variant clears the bit of an emptied price with `&= -bit` where the package has `&= ~bit`.

**Answer: (E).** `99 [(99, 100), (98, 0)]`.

**The integer.**
The bids are bits 4, 3 and 1 (F1): $16 + 8 + 2 = 26$, `0b11010`.

**The write.**
The withdrawal removes the whole 60 and writes `set_size(BUY, 98, 0)` (F4):
the list holds 0 at 98, and the bit is $2^3 = 8$.
By F3, $-8 = \neg 7$ has every bit from 3 upwards set and bits 0 to 2 clear, $\dots 111000$,
so $26 \;\&\; (-8) = 24$, `0b11000`.
Bit 3, the emptied price 98, stays set,
and bit 1, the price 96 that still holds 40, is cleared.
The package's $\neg 8 = -9$, $\dots 110111$, clears bit 3 alone (F3): $26 \;\&\; (-9) = 18$.

**The walk.**
`occupied_levels` takes the highest set bit of 24, bit 4, and records `(99, 100)`;
it clears that bit, takes bit 3, and records `(98, 0)`, the size the list holds there;
the integer is then 0, and the walk stops with two levels of the three asked for.
The best bid is the highest set bit, 99.

In [ ]:
26 & -8, -8 == ~7, ~8, 26 & ~8

**The other choices.**
- (A) `98 [(98, 0)]`: $-8$ read as 8, the sign taken apart from the bits:
  $26 \;\&\; 8 = 8$ keeps bit 3 alone.
- (B) `99 [(99, 100), (98, 0), (96, 40)]`: $-8$ taken for a mask with every bit set, as $-1$ is:
  the write clears nothing.
- (C) `99 [(99, 100), (96, 40)]`: the minus read as the tilde; this is what the package prints.
- (D) `99 [(99, 100)]`: the integer right, and `occupied_levels` taken to skip a price whose size is zero;
  it walks the set bits, and reads whatever the list holds at each.

In [ ]:
26 & 8, 26 & -1

## 11.5 · `question.gapsOfTheUncutBidOccupancy`

A `TickArrayBook` with floor 95 and width 12; bids at 104, 103, 100 and 99.
`count_binary_gaps`, the one-liner `bin(bits)[2:].count("10")` and `measure_largest_binary_gap`
are applied to the bid integer itself, not to its window.

**Answer: (A).** `1 2 4`.

**The integer.**
The bids are bits 9, 8, 5 and 4 (F1):
$$2^9 + 2^8 + 2^5 + 2^4 = 816,$$
`0b1100110000`.
Read from the top, it is a run of ones (104, 103), a gap of two zeros (102, 101),
a run of ones (100, 99), and four trailing zeros, the empty prices from 98 down to 95.
The trailing zeros lie below every one, so they are not a gap.

**`count_binary_gaps`.**
By Proposition A.7.2 (`prop.runsOfOnes`), $b \;\&\; \neg(2b)$ keeps one bit per maximal run of ones,
the lowest of each: here bits 8 and 4, `0b100010000`.
`n << 1` is $2b$, and the `&` with its complement is non-negative (F3).
Two runs make one gap: $2 - 1 = 1$.

**The one-liner.**
The string is `"1100110000"`, and `"10"` occurs wherever a one is followed by a zero:
at the top of the gap, and at the step from the lowest one into the trailing zeros.
It occurs twice: `2`.
It counts the gaps only on an odd integer.

**The flood.**
A pass `n |= n >> 1` sets every bit whose upper neighbour is set,
so it narrows by one bit, from the top, every run of zeros that lies below a one:
the gaps, Section A.8 (`sec.binaryInTheBook`), and the trailing zeros too,
as the docstring of `unito26.lob.binary_gaps.measure_largest_binary_gap` says.
The loop stops when $n \;\&\; (n + 1) = 0$, that is, when $n$ is a solid block of ones, $2^m - 1$,
since adding 1 flips the trailing ones and the lowest zero.
The gap of two closes in two passes,
and the four trailing zeros, filled from above one bit a pass as well, take four:

`0b1100110000`, `0b1110111000`, `0b1111111100`, `0b1111111110`, `0b1111111111`.

Four passes: `4`, the trailing zeros counted as a gap they are not.
The docstring of `unito26.lob.binary_gaps.measure_largest_binary_gap` requires an odd integer;
with a reported depth of four or more, `span_bits` shifts the window down to one first, `0b110011`,
on which the flood measures the gap of 2.

In [ ]:
b = 2**(104 - 95) + 2**(103 - 95) + 2**(100 - 95) + 2**(99 - 95)
n1 = b | b >> 1
n2 = n1 | n1 >> 1
n3 = n2 | n2 >> 1
n4 = n3 | n3 >> 1
b, bin(b), bin(b & ~(b << 1)), bin(n1), bin(n2), bin(n3), bin(n4), n3 & (n3 + 1), n4 & (n4 + 1)

**The other choices.**
- (B) `1 1 2`: the three read on the window of a reported depth of four or more, `0b110011`,
  the trailing zeros discarded as `span_bits` does;
  the snippet passes the uncut integer.
- (C) `1 2 2`: the flood taken to fill only the zeros between ones;
  it fills the trailing zeros from above as well.
- (D) `1 1 4`: the one-liner taken to count the gaps;
  it also counts the step into the trailing zeros.
- (E) `2 2 4`: `count_binary_gaps` taken to count the runs of ones, two;
  there is one gap fewer than there are runs.

## 11.6 · `question.askWindowsOfTheTwoBitmaps`

Asks 30 at 101, 50 at 102, 20 at 104 and 40 at 106,
in a `BitmapBook` with floor 95 and in a `TickArrayBook` with floor 95 and width 12;
the window of the three best asks, $L = 3$, in each.

**Answer: (D).** `0b1011 0b1101`.

**`BitmapBook`.**
It counts the asks up from the floor (F1), bits 6, 7, 9 and 11: the integer 2752, `0b101011000000`.
Its touch is the lowest set bit, and `span_bits` keeps the three lowest with `keep_lowest_set_bits`.
A pass of `rest &= rest - 1` clears the lowest set bit of `rest`:
subtracting 1 flips the trailing zeros and the lowest one and leaves the bits above it,
so the `&` clears that bit alone, as in the proof of Proposition A.5.2 (`prop.lowestSetBit`).
Three passes clear bits 6, 7 and 9 and leave $2^{11}$;
`n ^ rest` is the three bits cleared, `0b1011000000`;
and `discard_trailing_zeros` shifts it down by its six trailing zeros, to `0b1011`.
The touch, 101, is bit 0, and 104 is bit 3.

**`TickArrayBook`.**
It counts the asks down from the ceiling 106, bits 5, 4, 2 and 0: the integer 53, `0b110101`.
Its touch is the highest set bit, and `span_bits` keeps the three highest with `keep_highest_set_bits`.
By Proposition A.7.1 (`prop.rankByShift`), `(53 >> k).bit_count()` is the number of set bits
at positions $k$ and above, non-increasing in $k$,
and the third highest set bit is at the largest $k$ where it is at least 3.
53 has four set bits, more than 3, so the bisection runs: on $0 \leq k \leq 5$ it keeps that count at least 3 at `low`:
$k = 3$ counts 2, so `high = 2`; $k = 1$ counts 3, so `low = 1`; $k = 2$ counts 3, so `low = 2`.
The cut is at bit 2: `53 >> 2` is `0b1101`.
The deepest reported level, 104, is bit 0, and the touch, 101, is bit 3.

The two windows hold the same three asks, and each is the other read backwards.
Section A.7 (`sec.binaryRuns`) puts the deepest reported level at bit 0,
as `TickArrayBook` does on both sides;
`BitmapBook` cuts its asks at the low end, and puts the touch there.

In [ ]:
up = 2**(101 - 95) + 2**(102 - 95) + 2**(104 - 95) + 2**(106 - 95)
down = 2**(106 - 101) + 2**(106 - 102) + 2**(106 - 104) + 2**(106 - 106)
kept = up - 2**(106 - 95)
up, bin(up), down, bin(down), bin(kept), bin(kept >> 6), (down >> 3).bit_count(), (down >> 1).bit_count(), (down >> 2).bit_count(), bin(down >> 2)

**The other choices.**
- (A) Both `0b1101`: both books taken to hold their asks as `TickArrayBook` does,
  counted down from a ceiling and cut at the top;
  `BitmapBook` has no ceiling, and counts both sides up from the floor.
- (B) Both `0b1011`: both books taken to hold their asks as `BitmapBook` does,
  counted up from the floor and cut at the bottom.
- (C) `0b101011 0b110101`: the cut at three levels missed, all four asks kept and shifted down to an odd integer.
- (E) `0b1101 0b1011`: the two books exchanged, each taken to hold its asks as the other does.

A misreading of the count alone, or of the cut alone, gives `0b10101` for the book it concerns,
which is no choice.

## 11.7 · `question.lowestSetBitCostsTheBand`

Two asks, at 1 and at $W - 1$, in a `BitmapBook` with floor 0 and in a `TickArrayBook` with floor 0 and width $W$;
the time of one read of the best ask, for $W = 10^3, 10^4, 10^5, 10^6$.

**Answer: (A).**
A read of `array` takes about the same time at every width;
a read of `bitmap` grows with $W$, about in proportion to it at the larger widths,
and at $W = 10^6$ is many times slower.

**What each read computes.**
In `bitmap` the asks are bits 1 and $W - 1$ (F1),
an integer of $W$ bits whose touch is its lowest set bit,
read as `(bits & -bits).bit_length() - 1` (F2).
In `array` they are bits $(W - 1) - 1 = W - 2$ and 0, and the touch is the highest set bit,
read as `bits.bit_length() - 1`.
Both return the price 1.

**What each costs.**
CPython stores a non-negative integer as its digits in base $2^{30}$, together with their number.
`bit_length` reads that number and the highest digit, in $O(1)$.
The negation $b \mapsto -b$ and the `&` each build a new integer digit by digit, in $O(W)$,
wherever the lowest set bit lies, Section A.6 (`sec.binaryCosts`).
An integer of $W$ bits has $\lceil W/30 \rceil$ digits:
34 at $W = 10^3$, where the fixed cost of the call is comparable to the work on the digits,
and $33\,334$ at $W = 10^6$, where the work on the digits dominates.
Counting the asks down from the ceiling makes the best ask the highest set bit,
and spares the ask side this cost at every lookup, Section 1.2.4 (`sec.fasterBook`).

In the run of the verification notebook,
the read of `bitmap` grows about tenfold from $10^5$ to $10^6$
and is several hundred times slower than that of `array` at $10^6$,
whose time does not grow.

In [ ]:
ceil(10**3 / 30), ceil(10**4 / 30), ceil(10**5 / 30), ceil(10**6 / 30)

**The other choices.**
- (B) Both constant: each read taken for a fixed number of operations, as on a machine word,
  where both are single instructions;
  a Python integer of $W$ bits is an array of digits, and the negation and `&` visit every one.
- (C) Both in proportion to $W$: `bit_length` taken to scan the digits, as `&` does;
  it reads their stored number.
- (D) `bitmap` constant and `array` growing: the cost taken to depend on where the bit lies.
  `bit_length` costs the same wherever the highest bit is,
  and `bits & -bits` builds all $W$ bits wherever the lowest is.